# Cell2Fire-W — Belgium validation workshop

Simulate four **real 2025 Belgian wildfires** and compare each simulated fire scar
against the **observed perimeter**, sweeping fuel-moisture scenarios to find the
best fit. Goal: show Cell2Fire-W can reproduce observed fire spread in Belgium.

Each instance is centred on the real ignition; the observed scar lives in
`ground_truth/observed_scar.shp`.

## 0. Setup  (run me first)

In [ ]:
# === CONFIG: where to get the instances and the engine =====================
C2FW_REPO     = "https://github.com/fire2a/C2F-W"
C2FW_BRANCH   = "main"
INSTANCES_PATH = "data/handson/Instances_Belgium"      # instances live inside the repo

# Engine source. The v2.0 (continuous-moisture) engine is required for --moisture-scenario.
#   "drive"  -> download a prebuilt source zip from Google Drive (works on Colab today)
#   "github" -> build from a branch of C2F-W that already contains the v2.0 engine (preferred)
ENGINE_SOURCE   = "drive"
ENGINE_BRANCH   = "feat/sb-continuous-moisture"        # used only if ENGINE_SOURCE == "github"
SRC_DRIVE_ID    = "1bpt0EUadMi-d97I5_Ix-IQoFm_MswkAi"  # c2fw_v2_src.zip (used if "drive")
# ===========================================================================
import os, sys, subprocess, shutil, zipfile
from pathlib import Path
MAT = Path.cwd()/"c2f_be"; MAT.mkdir(exist_ok=True)

if shutil.which("g++") is None or shutil.which("make") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y g++ make libtiff-dev libboost-random-dev",
                   shell=True, check=False)

# 1) instances: sparse-checkout just the Instances_Belgium folder from the repo
repo = MAT/"C2F-W"
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--filter=blob:none","--sparse",
                    "-b",C2FW_BRANCH,C2FW_REPO,str(repo)],check=True)
    subprocess.run(["git","sparse-checkout","set",INSTANCES_PATH],cwd=str(repo),check=True)
BE = repo/INSTANCES_PATH
CASES = sorted([p.name for p in BE.iterdir() if p.is_dir()])
print("instances from", C2FW_REPO, "->", CASES)

# 2) engine
if ENGINE_SOURCE == "github":
    eng = MAT/"engine"
    if not eng.exists():
        subprocess.run(["git","clone","--depth","1","--filter=blob:none","--sparse",
                        "-b",ENGINE_BRANCH,C2FW_REPO,str(eng)],check=True)
        subprocess.run(["git","sparse-checkout","set","Cell2Fire"],cwd=str(eng),check=True)
    SRC_DIR = eng/"Cell2Fire"
else:
    subprocess.run([sys.executable,"-m","pip","install","-q","gdown"]); import gdown
    gdown.download(id=SRC_DRIVE_ID, output=str(MAT/"src.zip"), quiet=False)
    with zipfile.ZipFile(MAT/"src.zip") as z: z.extractall(MAT)
    SRC_DIR = MAT/"Cell2Fire"
subprocess.run(["make","clean"],cwd=str(SRC_DIR),check=False)
subprocess.run(["make"],cwd=str(SRC_DIR),check=True)
EXEC = str(SRC_DIR/"Cell2Fire"); os.environ["C2F_EXEC"]=EXEC
print("engine:", EXEC, "| exists:", Path(EXEC).exists())

In [ ]:
# python deps (Colab has most; pyshp is the robust shapefile reader we need)
import importlib, subprocess, sys
for pkg,mod in [("pyshp","shapefile"),("folium","folium"),("pyproj","pyproj"),("matplotlib","matplotlib")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pkg])
import numpy as np, matplotlib.pyplot as plt, csv, shutil, glob, subprocess
from matplotlib.colors import ListedColormap
from matplotlib.path import Path as MplPath
import shapefile
print("deps ready")

## 1. Choose a case
Everything downstream auto-derives from the chosen instance's `.asc` header —
grid size, cellsize, CRS and the centre-ignition cell. Switch `CASE` to re-run any site.

In [ ]:
CASE = "Case1_Clinge"        # Case1_Clinge | Case2_Houffalize | Case3_Arlon | Case4_Maasmechelen
INST = BE/CASE
WORK = Path("/tmp/c2f_be_runs")/CASE

def read_asc(path):
    header={}
    with open(path) as f:
        for _ in range(6):
            k,v=f.readline().split(); header[k.lower()]=float(v)
    data=np.loadtxt(path,skiprows=6)
    return np.ma.masked_equal(data,header.get("nodata_value",-9999)), header

fuels,h = read_asc(INST/"fuels.asc")
NCOLS,NROWS = int(h["ncols"]),int(h["nrows"])
CENTER = (NROWS//2)*NCOLS + NCOLS//2 + 1     # centre cell id (1-based)
CRS = "EPSG:3812"                             # Belge Lambert 2008 (all Belgium cases)
print(f"{CASE}: {NCOLS}x{NROWS} | cell {h['cellsize']:.0f} m | centre cell {CENTER} | CRS {CRS}")

## 2. Helpers (engine call + fuel colours)

In [ ]:
def set_ignition(cell):
    (INST/"Ignitions.csv").write_text(f"Year,Ncell\n1,{int(cell)}\n")

def run_c2f(tag, scenario="D2L2", *, max_periods=100, seed=1, outputs=("final-grid",), extra=None):
    out = WORK/"runs"/tag
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    cmd=[EXEC,"--input-instance-folder",str(INST)+"/","--output-folder",str(out)+"/",
         "--sim","S","--ignitions","--seed",str(seed),"--nsims","1","--nweathers","1",
         "--Fire-Period-Length","5.0","--Weather-Period-Length","60",
         "--max-fire-periods",str(max_periods),"--moisture-scenario",scenario]
    for o in outputs: cmd+=["--"+o]
    if extra: cmd+=extra
    r=subprocess.run(cmd,capture_output=True,text=True)
    if r.returncode!=0:
        print(r.stdout[-1200:]); print(r.stderr[-600:])
        raise RuntimeError(f"Cell2Fire failed (tag={tag})")
    return out

def final_scar(out, sim=1):
    gdir=out/"Grids"/f"Grids{sim}"
    g=sorted(gdir.glob("ForestGrid*.csv"),key=lambda p:int(p.stem.replace("ForestGrid","")))
    return np.loadtxt(g[-1],delimiter=",").astype(bool)

def fuel_rgb(fuels):
    lut={}
    lp=INST/"spain_lookup_table.csv"
    if lp.exists():
        for row in csv.DictReader(open(lp)):
            try: lut[int(float(row["grid_value"]))]=(int(row[" r"])/255,int(row[" g"])/255,int(row[" b"])/255)
            except: pass
    img=np.full(fuels.shape+(3,),0.85)
    for v in np.unique(fuels.filled(-1)):
        if int(v) in lut: img[fuels.filled(-1)==v]=lut[int(v)]
    return img
print("helpers ready")

## 3. The observed fire scar
Read the mapped perimeter (`ground_truth/observed_scar.shp`) and rasterise it onto
the instance grid. `pyshp` + `encodingErrors='replace'` tolerates non-UTF8 `.dbf` fields.

In [ ]:
def rasterize_perimeter(shp_path, header):
    try:    r=shapefile.Reader(str(shp_path), encodingErrors="replace")
    except Exception:
        sp=str(shp_path); r=shapefile.Reader(shp=open(sp,"rb"), shx=open(sp.replace(".shp",".shx"),"rb"))
    nc,nr=int(header["ncols"]),int(header["nrows"])
    xll,yll,cs=header["xllcorner"],header["yllcorner"],header["cellsize"]
    xs=xll+(np.arange(nc)+0.5)*cs; ys=yll+(nr-np.arange(nr)-0.5)*cs
    XX,YY=np.meshgrid(xs,ys); pts=np.column_stack([XX.ravel(),YY.ravel()])
    inside=np.zeros(pts.shape[0],bool)
    for sh in r.shapes():
        P=sh.points; parts=list(sh.parts)+[len(P)]; verts=[]; codes=[]
        for k in range(len(parts)-1):
            ring=P[parts[k]:parts[k+1]]
            if len(ring)<3: continue
            verts+=ring; codes+=[MplPath.MOVETO]+[MplPath.LINETO]*(len(ring)-2)+[MplPath.CLOSEPOLY]
        if verts: inside |= MplPath(verts,codes).contains_points(pts)
    return inside.reshape(nr,nc)

OBS = rasterize_perimeter(INST/"ground_truth/observed_scar.shp", h)
print(f"observed burned cells: {int(OBS.sum())}  (~{OBS.sum()*h['cellsize']**2/1e4:.2f} ha)")

plt.figure(figsize=(6,6)); plt.imshow(fuel_rgb(fuels))
plt.imshow(np.ma.masked_where(~OBS,OBS), cmap=ListedColormap(["#1f78ff"]), alpha=0.55)
plt.plot(NCOLS//2, NROWS//2, marker="*", ms=14, color="#FFD23F", mec="k")
plt.title(f"{CASE}: observed scar (blue) + ignition"); plt.xticks([]); plt.yticks([]); plt.show()

## 4. Agreement metrics & a first comparison

In [ ]:
def scar_metrics(sim, obs):
    sim=sim.astype(bool); obs=obs.astype(bool)
    TP=int((sim&obs).sum()); FP=int((sim&~obs).sum()); FN=int((~sim&obs).sum())
    cell_ha=h['cellsize']**2/1e4
    return dict(IoU=TP/(TP+FP+FN) if TP+FP+FN else 0, Dice=2*TP/(2*TP+FP+FN) if 2*TP+FP+FN else 0,
                POD=TP/(TP+FN) if TP+FN else 0, FAR=FP/(TP+FP) if TP+FP else 0,
                sim_ha=sim.sum()*cell_ha, obs_ha=obs.sum()*cell_ha)

def compare_plot(sim, obs, title):
    img=np.zeros(sim.shape+(4,))
    img[obs&~sim]=[0.12,0.47,1,0.75]      # missed (obs only) - blue
    img[sim&~obs]=[0.85,0.16,0.12,0.7]    # over-pred (sim only) - red
    img[sim&obs ]=[0.15,0.65,0.20,0.85]   # hit - green
    plt.figure(figsize=(6.2,6.2)); plt.imshow(fuel_rgb(fuels))
    plt.imshow(img); plt.plot(NCOLS//2,NROWS//2,marker="*",ms=13,color="#FFD23F",mec="k")
    m=scar_metrics(sim,obs)
    plt.title(f"{title}\nIoU={m['IoU']:.3f} Dice={m['Dice']:.3f} POD={m['POD']:.2f} FAR={m['FAR']:.2f}")
    plt.xticks([]); plt.yticks([]); plt.show()
    return m

set_ignition(CENTER)
sim = final_scar(run_c2f("first","D2L2"))
_ = compare_plot(sim, OBS, f"{CASE} — D2L2  (green=hit, blue=missed, red=over-predicted)")

## 5. Moisture sweep — find the best-fitting scenario
Wind is known (observed series in `Weather.csv`); fuel moisture is the unknown.
Sweep the 16 `DkLm` combinations and rank by IoU against the observed scar.

In [ ]:
set_ignition(CENTER)
rows=[]; IoU=np.zeros((4,4))
for dk in range(1,5):
    for lm in range(1,5):
        sc=f"D{dk}L{lm}"
        m=scar_metrics(final_scar(run_c2f(sc,sc)), OBS)
        IoU[dk-1,lm-1]=m["IoU"]; rows.append((sc,m))
rows.sort(key=lambda r:-r[1]["IoU"])
print(f"{'scenario':9}{'IoU':>7}{'Dice':>7}{'POD':>7}{'FAR':>7}{'sim_ha':>8}{'obs_ha':>8}")
for sc,m in rows:
    print(f"{sc:9}{m['IoU']:7.3f}{m['Dice']:7.3f}{m['POD']:7.3f}{m['FAR']:7.3f}{m['sim_ha']:8.2f}{m['obs_ha']:8.2f}")
BEST=rows[0][0]; print("\nBEST FIT:", BEST, f"(IoU={rows[0][1]['IoU']:.3f})")

fig,ax=plt.subplots(figsize=(5.6,4.8)); im=ax.imshow(IoU,cmap="viridis",origin="upper",vmin=0)
ax.set_xticks(range(4),[f"L{m}" for m in range(1,5)]); ax.set_yticks(range(4),[f"D{k}" for k in range(1,5)])
ax.set_xlabel("live moisture"); ax.set_ylabel("dead moisture"); ax.set_title(f"{CASE}: IoU vs observed scar")
for k in range(4):
    for mm in range(4):
        ax.text(mm,k,f"{IoU[k,mm]:.2f}",ha="center",va="center",
                color="white" if IoU[k,mm]<IoU.max()*0.6 else "black",fontsize=9)
plt.colorbar(im,shrink=.85); plt.tight_layout(); plt.show()

## 6. Best-fit comparison

In [ ]:
set_ignition(CENTER)
sim_best=final_scar(run_c2f("best",BEST))
m=compare_plot(sim_best, OBS, f"{CASE} — best fit {BEST}")
print(f"{CASE}: best scenario {BEST} -> matched {100*m['POD']:.0f}% of observed area, "
      f"over-predicted {100*m['FAR']:.0f}%, sim {m['sim_ha']:.1f} ha vs observed {m['obs_ha']:.1f} ha")

## 7. Project best fit on the real map

In [ ]:
import folium, pyproj
to_wgs=pyproj.Transformer.from_crs(CRS,"EPSG:4326",always_xy=True)
def bounds(hd):
    x0,y0=hd["xllcorner"],hd["yllcorner"]; x1=x0+hd["ncols"]*hd["cellsize"]; y1=y0+hd["nrows"]*hd["cellsize"]
    la,lo=[],[]
    for x in (x0,x1):
        for y in (y0,y1):
            o,a=to_wgs.transform(x,y); lo.append(o); la.append(a)
    return [[min(la),min(lo)],[max(la),max(lo)]]
def rgba(mask,color,a=160):
    o=np.zeros(mask.shape+(4,),np.uint8); o[mask]=(*color,a); return o
b=bounds(h); c=[(b[0][0]+b[1][0])/2,(b[0][1]+b[1][1])/2]
fmap=folium.Map(location=c,zoom_start=15,tiles=None,control_scale=True)
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 name="satellite",attr="Esri").add_to(fmap)
folium.TileLayer("OpenStreetMap",name="OSM").add_to(fmap)
folium.raster_layers.ImageOverlay(rgba(OBS,(31,120,255),150),bounds=b,name="observed").add_to(fmap)
folium.raster_layers.ImageOverlay(rgba(sim_best,(220,40,30),140),bounds=b,name=f"simulated {BEST}").add_to(fmap)
folium.LayerControl(collapsed=False).add_to(fmap); fmap.fit_bounds(b); fmap

## 8. Dynamic fire projection on the real map  (one cell per case)
For each Belgian fire: the **observed perimeter** (red outline) plus the **Cell2Fire
spread animated over time**, draped on a satellite basemap, using each case's best-fit
moisture. The bright **elliptical wavelets at the advancing front are the Huygens fire
fronts** — one ellipse per leading cell, rear focus at the cell, elongated downwind with
eccentricity set by the local rate of spread. Drag/zoom the map.

In [ ]:
import io, base64, math
from PIL import Image
from matplotlib.patches import Ellipse as _Ellipse
import folium, pyproj

# best-fit moisture per case (from the §5 sweep); tweak freely
BEST_BY_CASE = {"Case1_Clinge":"D4L2","Case2_Houffalize":"D1L3",
                "Case3_Arlon":"D2L2","Case4_Maasmechelen":"D4L3"}

def _arrival_grid(msg_csv, center, nc, nr):
    arcs=[(int(r[0]),int(r[1]),float(r[2])) for r in csv.reader(open(msg_csv)) if len(r)>=4]
    ign={center:0.0}
    for i,j,tj in arcs: ign[j]=min(ign.get(j,1e18),tj)
    T=np.full((nr,nc),np.nan)
    for c,t in ign.items(): T[(c-1)//nc,(c-1)%nc]=t
    return T

def _huygens_gif(out, center, hc, crop, frames_n=46, hold=8, W=6.6, wlen=1.0, lw=1.7):
    # Forward Huygens wavelets at the advancing front: one ellipse per leading-edge cell,
    # rear focus at the cell, oriented downwind (source->cell), eccentricity from ROS.
    NCOLS,NROWS=int(hc["ncols"]),int(hc["nrows"]); r0,r1,c0,c1=crop
    arcs=[(int(r[0]),int(r[1]),float(r[2]),float(r[3]))
          for r in csv.reader(open(out/"Messages"/"MessagesFile1.csv")) if len(r)>=4]
    Tj={center:0.0}; src={}
    for i,j,tj,ros in arcs:
        if tj<Tj.get(j,1e18): Tj[j]=tj; src[j]=(i,ros)
    rv=[a[3] for a in arcs]; rlo,rhi=min(rv),max(rv)
    Tig=np.full((NROWS,NCOLS),np.nan)
    for c,ti in Tj.items(): Tig[(c-1)//NCOLS,(c-1)%NCOLS]=ti
    def xy(c): rr=(c-1)//NCOLS; cc=(c-1)%NCOLS; return cc+0.5, NROWS-rr-0.5
    def ecc(ros): return 0.45+0.45*(ros-rlo)/(rhi-rlo+1e-9)
    DARK=np.array([0x33,0x22,0x1b])/255
    tmax=np.nanmax(Tig); T=tmax*1.04; step=T/frames_n; band=step*5.0
    times=list(np.linspace(0,T,frames_n))+[T]*hold
    fig=plt.figure(figsize=(W,W*(r1-r0)/(c1-c0)),dpi=120); ax=fig.add_axes([0,0,1,1]); fig.patch.set_alpha(0)
    def burned(t):
        b=(Tig<=t); img=np.zeros((NROWS,NCOLS,4)); img[b,:3]=DARK; img[b,3]=0.80; return img
    def draw(t):
        ax.cla(); ax.set_xlim(c0,c1); ax.set_ylim(NROWS-r1,NROWS-r0); ax.axis("off"); ax.patch.set_alpha(0)
        ax.imshow(burned(t),extent=[0,NCOLS,0,NROWS],origin="upper",interpolation="nearest",zorder=2)
        for c in [c for c,ti in Tj.items() if (t-band)<=ti<=t and c in src]:
            i,ros=src[c]; xc,yc=xy(c); xi,yi=xy(i)
            ang=math.atan2(yc-yi,xc-xi) if (xc!=xi or yc!=yi) else 0.0
            e=ecc(ros); L=wlen*(0.6+0.8*(ros-rlo)/(rhi-rlo+1e-9)); bb=L*math.sqrt(1-e*e); foc=L*e
            glow=min(1.0,(t-Tj[c])/max(band,1e-6))
            ax.add_patch(_Ellipse((xc+foc*math.cos(ang),yc+foc*math.sin(ang)),2*L,2*bb,
                angle=math.degrees(ang),facecolor=(1.0,0.55,0.12,0.18),
                edgecolor=(1.0,0.78-0.35*glow,0.18,0.95),lw=lw,zorder=4,antialiased=True))
    rgba=[]
    for t in times:
        draw(t); fig.canvas.draw(); rgba.append(np.asarray(fig.canvas.buffer_rgba()).copy())
    plt.close(fig)
    pil=[Image.fromarray(f,"RGBA") for f in rgba]
    master=pil[len(pil)//2].convert("RGB").quantize(colors=255,method=Image.MEDIANCUT)
    frames=[]
    for im in pil:
        a=im.getchannel("A"); q=im.convert("RGB").quantize(palette=master,dither=Image.NONE)
        q.paste(255,a.point(lambda v:255 if v<55 else 0)); q.info["transparency"]=255; frames.append(q)
    buf=io.BytesIO()
    frames[0].save(buf,format="GIF",save_all=True,append_images=frames[1:],
                   duration=120,loop=0,transparency=255,disposal=2,optimize=False)
    return buf.getvalue()

def _bounds_crop(hd, crop, crs="EPSG:3812"):
    tw=pyproj.Transformer.from_crs(crs,"EPSG:4326",always_xy=True)
    xll,yll,cs=hd["xllcorner"],hd["yllcorner"],hd["cellsize"]; nr=int(hd["nrows"])
    r0,r1,c0,c1=crop; x0=xll+c0*cs; x1=xll+c1*cs; y1=yll+(nr-r0)*cs; y0=yll+(nr-r1)*cs
    la,lo=[],[]
    for x in (x0,x1):
        for y in (y0,y1):
            o,a=tw.transform(x,y); lo.append(o); la.append(a)
    return [[min(la),min(lo)],[max(la),max(lo)]]

def _perimeter_latlon(shp_path, crs="EPSG:3812"):
    tw=pyproj.Transformer.from_crs(crs,"EPSG:4326",always_xy=True)
    try: r=shapefile.Reader(str(shp_path),encodingErrors="replace")
    except Exception:
        sp=str(shp_path); r=shapefile.Reader(shp=open(sp,"rb"),shx=open(sp.replace(".shp",".shx"),"rb"))
    rings=[]
    for sh in r.shapes():
        P=sh.points; parts=list(sh.parts)+[len(P)]
        for k in range(len(parts)-1):
            rings.append([[a,o] for (o,a) in (tw.transform(x,y) for x,y in P[parts[k]:parts[k+1]])])
    return rings

def project_fire(case, scenario=None, margin=12):
    inst=BE/case; _,hc=read_asc(inst/"fuels.asc")
    nc,nr=int(hc["ncols"]),int(hc["nrows"]); center=(nr//2)*nc+nc//2+1
    sc=scenario or BEST_BY_CASE.get(case,"D2L2")
    inst.joinpath("Ignitions.csv").write_text(f"Year,Ncell\n1,{center}\n")
    out=Path("/tmp/proj")/case
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    subprocess.run([EXEC,"--input-instance-folder",str(inst)+"/","--output-folder",str(out)+"/",
        "--sim","S","--ignitions","--seed","1","--nsims","1","--final-grid","--output-messages",
        "--moisture-scenario",sc,"--Fire-Period-Length","5.0","--Weather-Period-Length","60",
        "--max-fire-periods","160"],capture_output=True,text=True)
    T=_arrival_grid(out/"Messages/MessagesFile1.csv",center,nc,nr)
    rs,csx=np.where(np.isfinite(T))
    r0,r1=max(rs.min()-margin,0),min(rs.max()+margin,nr)
    c0,c1=max(csx.min()-margin,0),min(csx.max()+margin,nc)
    gif=_huygens_gif(out,center,hc,(r0,r1,c0,c1)); uri="data:image/gif;base64,"+base64.b64encode(gif).decode()
    b=_bounds_crop(hc,(r0,r1,c0,c1)); ctr=[(b[0][0]+b[1][0])/2,(b[0][1]+b[1][1])/2]
    fmap=folium.Map(location=ctr,zoom_start=16,tiles=None,control_scale=True)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     name="satellite",attr="Esri").add_to(fmap)
    folium.TileLayer("OpenStreetMap",name="OSM").add_to(fmap)
    for ring in _perimeter_latlon(inst/"ground_truth/observed_scar.shp"):
        folium.PolyLine(ring,color="#ff2d2d",weight=3,opacity=0.95,
                        tooltip="observed perimeter").add_to(fmap)
    folium.raster_layers.ImageOverlay(uri,bounds=b,opacity=0.92,
                                      name=f"Cell2Fire spread ({sc})",zindex=5).add_to(fmap)
    o,a=pyproj.Transformer.from_crs("EPSG:3812","EPSG:4326",always_xy=True).transform(
        hc["xllcorner"]+(center-1)%nc*hc["cellsize"]+hc["cellsize"]/2,
        hc["yllcorner"]+(nr-(center-1)//nc-0.5)*hc["cellsize"])
    folium.Marker([a,o],tooltip=f"ignition (cell {center})",
                  icon=folium.Icon(color="orange",icon="fire",prefix="fa")).add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap); fmap.fit_bounds(b)
    print(f"{case}: scenario {sc} | tmax={np.nanmax(T):.0f} min | burned cells={int(np.isfinite(T).sum())}")
    return fmap
print("projection helper ready")

### Case 1 — Clinge (coastal Flanders)

In [ ]:
project_fire("Case1_Clinge")

### Case 2 — Houffalize (Ardennes)

In [ ]:
project_fire("Case2_Houffalize")

### Case 3 — Arlon

In [ ]:
project_fire("Case3_Arlon")

### Case 4 — Maasmechelen (Limburg)

In [ ]:
project_fire("Case4_Maasmechelen")

## 9. Next steps
- Run all four cases (change `CASE` in §1) and tabulate best-fit IoU per site.
- The moisture scenario that best fits is the calibration result to report.
- To enable crown fire / EMC conditioning, supply real `ccf/cbd/cbh` layers and
  a `Weather.csv` with `T,RH` columns, then use `--moisture-mode conditioning`.